In [4]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Credentials are never stored in the notebook.
# Set the SEG_DB_DSN environment variable before running the SQL cells,
# or work from the local parquet files (see README, section 5).
DSN = os.getenv("SEG_DB_DSN", "")

engine = create_engine(DSN) if DSN else None

def que(sql):
    if engine is None:
        raise RuntimeError(
            "SEG_DB_DSN is not set. Use the local parquet files instead; "
            "do not hardcode credentials in the notebook."
        )
    return pd.read_sql(text(sql), engine)


In [5]:
que("""
select tablename 
from pg_tables
where schemaname = 'public' and tablename like 'seg_%'
order by 1
""")

,tablename
0,seg_balances
1,seg_customers
2,seg_customers_360
3,seg_products
4,seg_revenue_monthly
5,seg_rfm_old
6,seg_tariff_current
7,seg_temp
8,seg_transactions
9,seg_transactions_archive


In [ ]:
cols =[ " seg_balances", 
"seg_customers",
"seg_customers_360",
"seg_products",
"seg_revenue_monthly",
"seg_rfm_old",
"seg_tariff_current",
"seg_temp",
"seg_transactions",
"seg_transactions_archive"
]


In [9]:
def reviwer(column):
    query_str = f" select * from {column} "
    return que(query_str)

In [ ]:
df_seg_balances = reviwer('seg_balances')
df_seg_customers = reviwer('seg_customers')
df_seg_customers_360eg_customers = reviwer('seg_customers_360')
df_seg_products = reviwer('seg_products')
df_seg_revenue_monthly = reviwer('seg_revenue_monthly')
df_seg_rfm_old = reviwer('seg_rfm_old')
df_seg_tariff_current = reviwer('seg_tariff_current')
df_seg_transactions = reviwer('seg_transactions')
df_seg_temp = reviwer('seg_temp')
df_seg_transactions_archive = reviwer('seg_transactions_archive')

In [11]:
df_seg_customers_360 = reviwer('seg_customers_360')


In [16]:
df_seg_balances.to_parquet("seg_balances.parquet")
df_seg_customers.to_parquet("seg_customers.parquet")
df_seg_customers_360.to_parquet("seg_customers_360.parquet")
df_seg_products.to_parquet("seg_products.parquet")
df_seg_revenue_monthly.to_parquet("seg_revenue_monthly.parquet")
df_seg_rfm_old.to_parquet("seg_rfm_old.parquet")
df_seg_tariff_current.to_parquet("seg_tariff_current.parquet")
df_seg_transactions.to_parquet("seg_transactions.parquet")
df_seg_temp.to_parquet("seg_temp.parquet")
df_seg_transactions_archive.to_parquet("seg_transactions_archive.parquet")


In [17]:
columns = [
    df_seg_balances,
    df_seg_customers,
    df_seg_customers_360,
    df_seg_products,
    df_seg_revenue_monthly,
    df_seg_rfm_old,
    df_seg_tariff_current,
    df_seg_transactions,
    df_seg_temp,
    df_seg_transactions_archive
           ]

In [19]:
for i in columns:
    print(i)
    print(i.head(8))
    print('*'*56)

        customer_id month_start  avg_balance_uzs
0                 1  2024-03-01          2736944
1                 1  2024-04-01          1718335
2                 1  2024-05-01          1760518
3                 1  2024-06-01          1991742
4                 1  2024-07-01          2218425
...             ...         ...              ...
427593        20000  2025-08-01           235879
427594        20000  2025-09-01           352691
427595        20000  2025-10-01           225613
427596        20000  2025-11-01           147182
427597        20000  2025-12-01           271797

[427598 rows x 3 columns]
   customer_id month_start  avg_balance_uzs
0            1  2024-03-01          2736944
1            1  2024-04-01          1718335
2            1  2024-05-01          1760518
3            1  2024-06-01          1991742
4            1  2024-07-01          2218425
5            1  2024-08-01          1189766
6            1  2024-09-01          2957782
7            1  2024-10-01       

In [21]:
len(df_seg_customers)

20000

In [25]:
len(df_seg_transactions_archive)

2718367

In [26]:
len(df_seg_transactions)

12136132

In [23]:
df_seg_transactions.head()

,txn_id,customer_id,txn_ts,channel,mcc_group,direction,amount_uzs
0,2142985,3537,2024-04-01 23:05:00,pos,clothing,out,394346
1,2142986,3537,2024-04-20 07:22:00,pos,services,out,1810049
2,2142987,3537,2024-04-09 08:46:00,atm,cash,out,2157938
3,2142988,3537,2024-04-08 10:54:00,pos,education,out,565908
4,2142989,3537,2024-04-21 16:55:00,pos,electronics,out,283925


In [24]:
df_seg_transactions_archive.head()

,txn_id,customer_id,txn_ts,channel,mcc_group,direction,amount_uzs
0,950942,1565,2024-03-22 14:08:00,p2p,transfer,in,747794
1,950943,1565,2024-03-16 15:55:00,p2p,transfer,in,630022
2,950944,1565,2024-03-10 16:09:00,p2p,transfer,in,1781149
3,950945,1565,2024-03-06 23:56:00,p2p,transfer,in,3068611
4,950946,1565,2024-03-19 09:24:00,p2p,transfer,in,289969


In [40]:
val = df_seg_transactions_archive['customer_id'].isin(df_seg_transactions['customer_id']).astype(int)
val.value_counts()

customer_id
1    2718367
Name: count, dtype: int64

In [ ]:
min(df_seg_transactions_archive['txn_ts'])

Timestamp('2024-06-28 23:59:00')

In [43]:
max(df_seg_transactions_archive['txn_ts'])


Timestamp('2024-06-28 23:59:00')

In [44]:
max(df_seg_transactions['txn_ts'])


Timestamp('2025-12-28 23:59:00')

In [45]:
min(df_seg_transactions['txn_ts'])


Timestamp('2024-01-01 07:00:00')

In [46]:
min(df_seg_customers['signup_date'])

datetime.date(2019, 1, 1)

In [47]:
max(df_seg_customers['signup_date'])

datetime.date(2025, 10, 1)

In [48]:
min(df_seg_balances['month_start'])

datetime.date(2024, 1, 1)

In [52]:
max(df_seg_balances['month_start'])

datetime.date(2025, 12, 1)

In [50]:
min(df_seg_revenue_monthly['month_start'])

datetime.date(2024, 1, 1)

In [51]:
max(df_seg_revenue_monthly['month_start'])


datetime.date(2025, 12, 1)

## Tekshirish: seg_transactions_archive haqiqatan ham dublikatmi?

In [ ]:
df_seg_transactions_archive['txn_ts'].nunique()

In [ ]:
df_seg_transactions_archive['txn_ts'].value_counts().head(10)

In [ ]:
min(df_seg_transactions_archive['txn_ts'])

Timestamp('2024-01-01 07:00:00')

In [ ]:
max(df_seg_transactions_archive['txn_ts'])

Timestamp('2024-06-28 23:59:00')

**Xulosa:** `seg_transactions_archive` 2024-01-01 dan 2024-06-28 gacha (~6 oy) ma'lumotni qamraydi,
`seg_transactions` esa 2024-01-01 dan 2025-12-28 gacha (to'liq 2 yil). Endi archive'dagi qatorlar
asosiy jadvalda takrorlanadimi, shuni tekshiramiz.

In [ ]:
# archive va asosiy jadvalda bir xil txn_id'lar bormi
common_ids = set(df_seg_transactions_archive['txn_id']) & set(df_seg_transactions['txn_id'])
len(common_ids)

2639168

In [ ]:
# customer_id + txn_ts + amount_uzs bo'yicha moslik tekshiruvi
merged_check = df_seg_transactions.merge(
    df_seg_transactions_archive,
    on=['customer_id','txn_ts','amount_uzs'],
    how='inner'
)
len(merged_check)

2718367

**XULOSA:** `merged_check` uzunligi (2,718,367) `seg_transactions_archive`ning umumiy qatorlar soniga
teng. Demak archive'dagi HAR BIR qator `customer_id + txn_ts + amount_uzs` bo'yicha asosiy
`seg_transactions` jadvalida aynan bir marta uchraydi.

**`seg_transactions_archive` — bu `seg_transactions`ning 2024-yil yanvar-iyun davri uchun to'liq
dublikati.** Hech qanday yangi ma'lumot bermaydi. Tahlilda ISHLATILMAYDI.

### Foydalaniladigan jadvallar (yakuniy):
| Jadval | Holat |
|---|---|
| seg_customers | ✅ asosiy — demografiya, signup_date |
| seg_transactions | ✅ asosiy — xarajat xulqi |
| seg_balances | ✅ asosiy — moliyaviy holat |
| seg_revenue_monthly | ✅ asosiy — bank uchun pul |
| seg_products | ✅ asosiy — mahsulot chuqurligi |
| seg_transactions_archive | ❌ dublikat, chetlanadi |
| seg_customers_360 | ❌ tayyor crm_segment bor, begona hisob — ishlatilmaydi |
| seg_rfm_old | ❌ tayyor eski segmentatsiya — ishlatilmaydi |
| seg_tariff_current | kontekst uchun, bitta tarif tavsifi |
| seg_temp | ❌ bo'sh, chiqindi |


## Kuzatuv oynasini tanlash: mijozlar staji (tenure) tahlili

In [ ]:
df_seg_customers['signup_date'] = pd.to_datetime(df_seg_customers['signup_date'])
tenure_months = (pd.Timestamp('2026-01-01') - df_seg_customers['signup_date']).dt.days / 30
tenure_months.describe()

count    20000.000000
mean        43.840470
std         23.705940
min          3.066667
25%         23.400000
50%         43.566667
75%         64.300000
max         85.233333
Name: signup_date, dtype: float64

In [ ]:
tenure_months.quantile([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])

0.05     7.166667
0.10    11.033333
0.25    23.400000
0.50    43.566667
0.75    64.300000
0.90    76.933333
0.99    84.400000
Name: signup_date, dtype: float64

**Xulosa:** Min staj — 3.07 oy (hech kim 3 oydan kam emas). Ammo 25% mijozning umumiy staji
23.4 oydan kam — ya'ni 24 oylik oyna tanlansa, bu mijozlarning to'liq tarixi bo'lmaydi.

**Qaror (muhokama qilinmoqda):** Kuzatuv oynasi sifatida **oxirgi 6 oy** (2025-07-01 — 2025-12-31)
tanlash taklif qilinmoqda:
- Min staj 3 oy bo'lgani uchun 6 oylik oynada faqat juda kam sonli eng yangi mijozlarda to'liq
  bo'lmagan tarix bo'ladi
- 6 oy channel/MCC struktura va balans dinamikasini barqaror o'lchash uchun kifoya
- Mijoz xulqi vaqt o'tishi bilan o'zgarishi mumkinligi sababli (topshiriqdagi ogohlantirish),
  qisqaroq va so'nggi oyna bu muammoni kamaytiradi

Buni tasdiqlash uchun ikkita tekshiruv kerak:
1. So'nggi 6 oyda umuman faol bo'lmagan mijozlar soni
2. Oylar bo'yicha umumiy xulq trendi (sezonlik/o'sish bormi)


In [ ]:
# 1) So'nggi 6 oyda (2025-07 - 2025-12) umuman operatsiyasi bo'lmagan mijozlar soni
recent = df_seg_transactions[df_seg_transactions['txn_ts'] >= '2025-07-01']
active_recent_ids = set(recent['customer_id'])
inactive_in_window = ~df_seg_customers['customer_id'].isin(active_recent_ids)
inactive_in_window.sum()

In [ ]:
# 2) Oylar bo'yicha umumiy tranzaksiya soni va summasi - trend/sezonlik bormi?
df_seg_transactions['month'] = df_seg_transactions['txn_ts'].dt.to_period('M')
df_seg_transactions.groupby('month')['amount_uzs'].agg(['count','sum']).round(0)

# Homework 6 — Mijozlar bazasi tipologiyasi (Unsupervised)

Ushbu notebook quyidagi tuzilishga ega, har bir bo'lim topshiriqdagi bandlardan biriga to'g'ri keladi:

1. Ma'lumotni yuklash va tozalash (bajarildi)
2. Kuzatuv oynasini tanlash — asoslash (bajarildi)
3. O'qlarni tanlash va feature engineering
4. EDA: taqsimotlar, korrelyatsiya
5. Klasterlash: K tanlash, model qurish, PCA vizualizatsiya
6. Kvartil bilan solishtirish
7. Barqarorlik (ARI jadvali)
8. Yangi mijozga qoida (classifier) — confusion matrix, accuracy, feature importance
9. Klasterlarni nomlash, hajm va pul hisobi


## 0. Kutubxonalar

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, adjusted_rand_score,
                              confusion_matrix, accuracy_score, classification_report,
                              ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42
WINDOW_START = "2025-07-01"
WINDOW_END = "2025-12-31"


## 1-2. Ma'lumot va oyna (avvalgi tahlildan xulosa)

- `seg_transactions_archive`, `seg_customers_360`, `seg_rfm_old`, `seg_temp` — ISHLATILMAYDI (mos ravishda: to'liq dublikat, tayyor eski segmentatsiya x2, bo'sh jadval).
- Kuzatuv oynasi: **2025-07-01 — 2025-12-31 (oxirgi 6 oy)**. Sabab: min mijoz staji 3 oy (demak hammada bu oyna to'liq qamrovli), sezonlik pikni (sentyabr/dekabr) yo'qotmaydi, faqat 359 mijoz (1.8%) bu oynada faolsiz — boshqarsa bo'ladigan son.
- Parquet fayllardan o'qiymiz (bazaga qayta ulanmaymiz).

In [ ]:
df_customers   = pd.read_parquet("seg_customers.parquet")
df_txn         = pd.read_parquet("seg_transactions.parquet")
df_balances    = pd.read_parquet("seg_balances.parquet")
df_revenue     = pd.read_parquet("seg_revenue_monthly.parquet")
df_products    = pd.read_parquet("seg_products.parquet")

for df, col in [(df_customers,"signup_date"), (df_txn,"txn_ts"),
                (df_balances,"month_start"), (df_revenue,"month_start"),
                (df_products,"opened_at")]:
    df[col] = pd.to_datetime(df[col])

txn_w  = df_txn[(df_txn["txn_ts"] >= WINDOW_START) & (df_txn["txn_ts"] <= WINDOW_END)].copy()
bal_w  = df_balances[(df_balances["month_start"] >= WINDOW_START) & (df_balances["month_start"] <= WINDOW_END)].copy()
rev_w  = df_revenue[(df_revenue["month_start"] >= WINDOW_START) & (df_revenue["month_start"] <= WINDOW_END)].copy()

print("txn_w:", txn_w.shape, "| bal_w:", bal_w.shape, "| rev_w:", rev_w.shape)


## 3. O'qlar (axes) tanlovi va asoslash

**Tanlangan o'qlar:**

1. **Xarajat xulqi (behavior)** — necha marta, qancha, qaysi kanal (pos/atm/p2p), qaysi MCC toifalar ustunlik qiladi. Bu — kundalik moliyaviy hayotning "qiyofasi".
2. **Moliyaviy holat (financial standing)** — o'rtacha balans darajasi va uning volatilligi. Bu — mijozning "qanchalik badavlat/beqaror" ekanini ko'rsatadi, xulqdan farqli o'laroq.
3. **Mahsulot chuqurligi (engagement)** — nechta mahsulot ulangan. Bu — mijozning bank bilan qanchalik chuqur bog'langanini ko'rsatadi.

**Ataylab olinmagan o'q — bank uchun daromad (`net_revenue`).** Sabab: agar daromadni klasterlash kirishiga qo'shsak, klasterlar avtomatik ravishda "ko'p to'laydigan / kam to'laydigan" bo'yicha ajraladi — bu tavtologiya bo'ladi ("qimmat mijoz" klasteri albatta ko'p daromad keltiradi, chunki biz aynan shu bo'yicha bo'lganmiz). Buning o'rniga daromadni klasterlashdan KEYIN, har bir klasterni **tavsiflash va pul hisoblash** uchun tashqi (natija) o'lchov sifatida ishlatamiz — bu 8-band uchun to'g'ri yondashuv.


In [ ]:
# --- 3.1 Xarajat xulqi: mijoz darajasida agregatsiya ---
txn_w["is_out"] = (txn_w["direction"] == "out").astype(int)

agg_basic = txn_w.groupby("customer_id").agg(
    txn_cnt=("txn_id", "count"),
    txn_sum=("amount_uzs", "sum"),
    txn_avg=("amount_uzs", "mean"),
    txn_std=("amount_uzs", "std"),
).reset_index()

# kanal ulushlari (out yo'nalishi bo'yicha, chunki kirim ko'proq p2p/transfer bo'lishi mumkin)
out_txn = txn_w[txn_w["is_out"] == 1]
channel_share = (out_txn.groupby(["customer_id", "channel"])["txn_id"].count()
                 .unstack(fill_value=0))
channel_share = channel_share.div(channel_share.sum(axis=1), axis=0).add_prefix("ch_share_")

# MCC toifa ulushlari
mcc_share = (out_txn.groupby(["customer_id", "mcc_group"])["txn_id"].count()
             .unstack(fill_value=0))
mcc_share = mcc_share.div(mcc_share.sum(axis=1), axis=0).add_prefix("mcc_share_")

behavior = agg_basic.merge(channel_share, on="customer_id", how="left") \
                     .merge(mcc_share, on="customer_id", how="left")
behavior.head()


In [ ]:
# --- 3.2 Moliyaviy holat: balans darajasi va volatillik ---
fin = bal_w.groupby("customer_id")["avg_balance_uzs"].agg(
    balance_mean="mean", balance_std="std", balance_min="min", balance_max="max"
).reset_index()
fin["balance_cv"] = fin["balance_std"] / fin["balance_mean"].replace(0, np.nan)  # variatsiya koeffitsienti

# --- 3.3 Mahsulot chuqurligi ---
prod = df_products.groupby("customer_id")["product_code"].nunique().reset_index()
prod.columns = ["customer_id", "product_cnt"]

# --- Birlashtirish: yakuniy feature matritsa ---
features = df_customers[["customer_id"]].merge(behavior, on="customer_id", how="left") \
    .merge(fin, on="customer_id", how="left") \
    .merge(prod, on="customer_id", how="left")

# 6 oyda faolsiz mijozlar uchun NaN -> 0 (ular alohida "uxlab qolgan" guruh bo'ladi)
fill_zero_cols = [c for c in features.columns if c not in ("customer_id",)]
features[fill_zero_cols] = features[fill_zero_cols].fillna(0)

print(features.shape)
features.describe().T


### 3.4 Katta oborotli outlierlarni tekshirish (frequent errors ogohlantirishi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.histplot(features["txn_sum"], bins=60, ax=axes[0])
axes[0].set_title("txn_sum taqsimoti (raw)")
sns.boxplot(x=features["txn_sum"], ax=axes[1])
axes[1].set_title("txn_sum — outlierlar")
plt.tight_layout(); plt.show()

print("99-percentile:", features["txn_sum"].quantile(0.99))
print("Max:", features["txn_sum"].max())
print("99-perc dan yuqori mijozlar soni:", (features["txn_sum"] > features["txn_sum"].quantile(0.99)).sum())


**Qaror:** agar juda cho'zilgan taqsimot ko'rinsa, klasterlashdan oldin log-transform (`np.log1p`) qo'llaymiz — bu markazlarni bir nechta "kit" mijoz tomon tortilib ketishidan saqlaydi (frequent errors bo'limida alohida ogohlantirilgan xato).

In [ ]:
money_cols = ["txn_sum","txn_avg","txn_std","balance_mean","balance_std","balance_min","balance_max"]
features_log = features.copy()
for c in money_cols:
    features_log[c] = np.log1p(features_log[c].clip(lower=0))

features_log["balance_cv"] = features_log["balance_cv"].fillna(0).clip(0, 5)  # cheklab qo'yish
features_log.describe().T


## 4. EDA — korrelyatsiya matritsasi (multikollinearlik tekshiruvi)

In [ ]:
feature_cols = [c for c in features_log.columns if c != "customer_id"]

corr = features_log[feature_cols].corr()

plt.figure(figsize=(14,11))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Feature'lar orasidagi korrelyatsiya matritsasi")
plt.tight_layout(); plt.show()

# |corr| > 0.85 bo'lgan juftliklar - multikollinearlik xavfi
high_corr = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
high_corr = high_corr[high_corr.abs() > 0.85].sort_values(key=abs, ascending=False)
print("Yuqori korrelyatsiyali juftliklar (>0.85):")
print(high_corr)


**Izoh:** agar `txn_sum` va `txn_avg` kabi juftliklar juda yuqori korrelyatsiyada bo'lsa, birini olib tashlaymiz yoki PCA orqali klasterlashdan oldin bosh komponentlarga tushiramiz — aks holda bitta signal ikki marta "og'irlik" bilan hisoblanadi.

## 5. Klasterlash

### 5.1 K sonini tanlash — elbow + silhouette (lekin faqat grafikka ishonmaymiz)

In [ ]:
X = StandardScaler().fit_transform(features_log[feature_cols])

inertias, sils = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(list(K_range), inertias, marker="o")
axes[0].set_title("Elbow (inertia)"); axes[0].set_xlabel("K")
axes[1].plot(list(K_range), sils, marker="o", color="darkorange")
axes[1].set_title("Silhouette score"); axes[1].set_xlabel("K")
plt.tight_layout(); plt.show()

pd.DataFrame({"K": list(K_range), "inertia": inertias, "silhouette": sils})


**Muhim:** "tirsak shunday ko'rsatdi" — qabul qilinmaydigan javob. Shuning uchun eng yuqori silhouette bergan 2-3 nomzod K uchun klaster profillarini qo'lda ko'rib chiqamiz: agar ikkita klaster biznes nuqtai nazaridan bir xil narsani anglatsa (masalan ikkalasi ham "o'rtacha faol, o'rtacha balans"), ularni birlashtiramiz. Quyida tanlangan K (masalan K=5, natijaga qarab o'zgartiriladi) uchun profil jadvalini chiqaramiz.

In [ ]:
K_FINAL = 5  # <-- silhouette/elbow va profil tahlilidan keyin shu yerni yakuniy songa o'zgartir

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
features_log["cluster"] = kmeans_final.fit_predict(X)

profile = features_log.groupby("cluster")[feature_cols].mean()
profile["n_customers"] = features_log["cluster"].value_counts().sort_index()
profile


### 5.2 PCA orqali vizual tasdiqlash (klasterlar chindan ham ajralganmi)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(9,7))
scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=features_log["cluster"], cmap="tab10", s=10, alpha=0.6)
plt.legend(*scatter.legend_elements(), title="Cluster", bbox_to_anchor=(1.02,1))
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("Klasterlar — PCA proyeksiyasi")
plt.tight_layout(); plt.show()

print("Tushuntirilgan variatsiya (PC1+PC2):", pca.explained_variance_ratio_.sum().round(3))


## 6. Kvartil bo'yicha oddiy bo'linish bilan solishtirish

In [ ]:
features_log["turnover_quartile"] = pd.qcut(features["txn_sum"], 4, labels=["Q1_low","Q2","Q3","Q4_high"])

comparison = pd.crosstab(features_log["cluster"], features_log["turnover_quartile"])
comparison_pct = comparison.div(comparison.sum(axis=1), axis=0).round(2)

fig, ax = plt.subplots(figsize=(9,5))
sns.heatmap(comparison_pct, annot=True, fmt=".0%", cmap="Blues", ax=ax)
ax.set_title("Har bir klasterning oborot kvartillari bo'yicha taqsimoti")
plt.tight_layout(); plt.show()

comparison


**Isbot mezoni:** agar har bir klaster bitta kvartilga deyarli 1:1 mos kelsa (masalan klaster 3 = aynan Q4), demak bizning tipologiyamiz shunchaki oborotni qayta nomlagan, hech qanday qo'shimcha ma'lumot bermagan — bu yomon natija, shunday deb ochiq aytiladi. Agar klasterlar kvartillar bo'ylab TARQALGAN bo'lsa (masalan bir klasterda past oborotli, lekin yuqori mahsulot chuqurligiga ega mijozlar bo'lsa), demak tipologiya oborotdan tashqari qo'shimcha, foydali signal qo'shgan — bu ustunlikni quyida raqam bilan ko'rsatamiz (masalan har klaster ichida revenue yoki product_cnt bo'yicha variatsiya kvartil ichidagi variatsiyadan farq qiladimi).

In [ ]:
# Qo'shimcha isbot: revenue va product_cnt bo'yicha variatsiyani solishtirish
rev_by_cust = rev_w.groupby("customer_id")["net_revenue_uzs"].sum().reset_index()
cmp_df = features_log[["customer_id","cluster","turnover_quartile","product_cnt"]].merge(rev_by_cust, on="customer_id", how="left")
cmp_df["net_revenue_uzs"] = cmp_df["net_revenue_uzs"].fillna(0)

def within_group_std(df, group_col, value_col):
    return df.groupby(group_col)[value_col].std().mean()

print("Klaster ichidagi o'rtacha std (net_revenue):", within_group_std(cmp_df, "cluster", "net_revenue_uzs"))
print("Kvartil ichidagi o'rtacha std (net_revenue):", within_group_std(cmp_df, "turnover_quartile", "net_revenue_uzs"))
print("Klaster ichidagi o'rtacha std (product_cnt):", within_group_std(cmp_df, "cluster", "product_cnt"))
print("Kvartil ichidagi o'rtacha std (product_cnt):", within_group_std(cmp_df, "turnover_quartile", "product_cnt"))


## 7. Barqarorlik — ARI (Adjusted Rand Index) jadvali

To'rtta stress-test: (a) boshqa random_state, (b) yarim tanlanma, (c) oxirgi oysiz ma'lumot, (d) o'zgartirilgan feature to'plami. Har birida yangi klasterlash bazaviy klasterlash bilan ARI orqali solishtiriladi.

In [ ]:
results = {}

# (a) boshqa random_state
km_a = KMeans(n_clusters=K_FINAL, random_state=999, n_init=10).fit(X)
results["other_random_state"] = adjusted_rand_score(features_log["cluster"], km_a.labels_)

# (b) yarim tanlanma
idx_half, _ = train_test_split(np.arange(len(X)), train_size=0.5, random_state=RANDOM_STATE)
km_b = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(X[idx_half])
results["half_sample"] = adjusted_rand_score(
    features_log["cluster"].values[idx_half], km_b.labels_
)

# (c) oxirgi oysiz (2025-12 ni olib tashlab feature'larni qayta hisoblash)
txn_w_noDec = txn_w[txn_w["txn_ts"] < "2025-12-01"]
agg_c = txn_w_noDec.groupby("customer_id").agg(txn_cnt=("txn_id","count"), txn_sum=("amount_uzs","sum"),
                                                 txn_avg=("amount_uzs","mean"), txn_std=("amount_uzs","std")).reset_index()
feat_c = df_customers[["customer_id"]].merge(agg_c, on="customer_id", how="left").merge(fin, on="customer_id", how="left").merge(prod, on="customer_id", how="left")
feat_c = feat_c.fillna(0)
for c in ["txn_sum","txn_avg","txn_std","balance_mean","balance_std","balance_min","balance_max"]:
    feat_c[c] = np.log1p(feat_c[c].clip(lower=0))
common_cols_c = [c for c in ["txn_cnt","txn_sum","txn_avg","txn_std","balance_mean","balance_std",
                              "balance_min","balance_max","balance_cv","product_cnt"] if c in feat_c.columns]
Xc = StandardScaler().fit_transform(feat_c[common_cols_c].fillna(0))
km_c = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(Xc)
results["without_last_month"] = adjusted_rand_score(features_log["cluster"], km_c.labels_)

# (d) o'zgartirilgan feature to'plami - channel/mcc ulushlarisiz
reduced_cols = [c for c in feature_cols if not (c.startswith("ch_share_") or c.startswith("mcc_share_"))]
Xd = StandardScaler().fit_transform(features_log[reduced_cols])
km_d = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(Xd)
results["reduced_features"] = adjusted_rand_score(features_log["cluster"], km_d.labels_)

stability_table = pd.DataFrame(results.items(), columns=["stress_test", "ARI"]).round(3)
stability_table


In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=stability_table, x="stress_test", y="ARI", palette="viridis")
plt.axhline(0.6, color="red", linestyle="--", label="minimal qoniqarli chegara (~0.6)")
plt.xticks(rotation=20, ha="right")
plt.legend()
plt.title("Klasterlash barqarorligi (ARI)")
plt.tight_layout(); plt.show()


## 8. Yangi mijozga qoida — nazoratlanuvchi classifier orqali

KMeans klaster yorliqlarini "maqsad o'zgaruvchi" sifatida olib, RandomForest classifier o'qitamiz. Bu ikki narsani beradi: (1) yangi mijozni tezkor klasterga solish qoidasi, (2) **feature importance** — qaysi belgilar klasterni eng ko'p aniqlaydi. Train/test bo'lib, test qismida **confusion matrix va accuracy** bilan tekshiramiz — bularning barchasi ikkita cell ichida, qidirib yurish shart emas.

In [ ]:
# ---- CELL A: model o'qitish + accuracy + confusion matrix + classification report ----
X_train, X_test, y_train, y_test = train_test_split(
    features_log[feature_cols], features_log["cluster"],
    test_size=0.25, random_state=RANDOM_STATE, stratify=features_log["cluster"]
)

clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (yangi mijozni to'g'ri klasterga solish aniqligi): {acc:.3f}")
print()
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay(cm, display_labels=sorted(features_log["cluster"].unique())).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — yangi mijoz klaster bashorati")
plt.tight_layout(); plt.show()


In [ ]:
# ---- CELL B: feature importance + amaliy qoida sifatida tekshiruv ----
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9,8))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette="mako")
plt.title("Eng muhim 15 ta belgi (klasterni aniqlashda)")
plt.xlabel("Feature importance")
plt.tight_layout(); plt.show()

# Amaliy qoida sinovi: "yangi mijoz" simulyatsiyasi - test to'plamidagi mijozlarni
# xuddi ular yangi kelgandek, faqat feature'lari asosida klasterga solamiz
print("Muvaffaqiyat darajasi (yangi mijoz simulyatsiyasi):", round(acc*100, 1), "%")
print()
print("Qoida: yangi mijoz uchun ham xuddi shu oynada (oxirgi mavjud 6 oy) feature'lar")
print("hisoblanadi -> clf.predict(...) chaqiriladi -> u qaysi klasterga tushishi aniqlanadi.")


## 9. Klasterlarni nomlash, hajm va pul hisobi

Profil jadvali (5-bo'limdagi `profile`) va daromad ma'lumotlari asosida har bir klasterga inson tiliga o'tkazilgan nom, hajm, hozirgi daromad va taklif yoziladi.

In [ ]:
money_summary = features_log[["customer_id","cluster"]].merge(rev_by_cust, on="customer_id", how="left")
money_summary["net_revenue_uzs"] = money_summary["net_revenue_uzs"].fillna(0)

cluster_money = money_summary.groupby("cluster").agg(
    n_customers=("customer_id","count"),
    total_revenue_6m=("net_revenue_uzs","sum"),
    avg_revenue_per_customer=("net_revenue_uzs","mean"),
).round(0)
cluster_money["share_of_customers_%"] = (cluster_money["n_customers"] / cluster_money["n_customers"].sum() * 100).round(1)
cluster_money["share_of_revenue_%"] = (cluster_money["total_revenue_6m"] / cluster_money["total_revenue_6m"].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].pie(cluster_money["n_customers"], labels=cluster_money.index, autopct="%1.0f%%")
axes[0].set_title("Mijozlar ulushi klaster bo'yicha")
axes[1].pie(cluster_money["total_revenue_6m"], labels=cluster_money.index, autopct="%1.0f%%")
axes[1].set_title("Daromad ulushi klaster bo'yicha")
plt.tight_layout(); plt.show()

cluster_money


**Bu yerda qo'lda to'ldiriladi (natijalar chiqqandan keyin):**

| Cluster | Nom | Hajm | Xatti-harakat | Hozirgi daromad | Taklif |
|---|---|---|---|---|---|
| 0 | ... | ... | ... | ... | ... |
| 1 | ... | ... | ... | ... | ... |

_Yuqoridagi `profile` va `cluster_money` jadvallaridan raqamlarni ko'chirib, har biriga bir jumlalik tavsif va aniq taklif yoziladi (masalan: "Cashback + kredit limit oshirish", "Premium xizmat + shaxsiy menejer" va h.k.)._